In [19]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import pytz

In [20]:
# Load Playstore dataset
apps = pd.read_csv(r"C:\Users\vanda\Downloads\Play Store Data.csv")
apps.head()  

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [21]:
# Data Cleaning
apps["Rating"] = pd.to_numeric(apps["Rating"],errors = "coerce")

In [22]:
apps["Reviews"] =(apps["Reviews"].astype(str).str.replace(",","",regex = False))

In [23]:
apps["Reviews"] = pd.to_numeric(apps["Reviews"],errors = "coerce")

In [24]:
apps["Installs"] =(apps["Installs"].astype(str).str.replace(",","",regex = False).str.replace("+","",regex = False))

In [25]:
apps["Installs"] = pd.to_numeric(apps["Installs"],errors = "coerce")

In [26]:
# Convert Size 
apps =apps[apps["Size"].str.contains("M",na = False)]

In [27]:
apps["Size"] = (apps["Size"].str.replace("M","",regex = False).str.replace(",","",regex = False))

In [28]:
apps["Size"] = pd.to_numeric(apps["Size"],errors = "coerce")

In [29]:
# Convert Last updated
apps["Last Updated"] = pd.to_datetime(apps["Last Updated"],errors = "coerce")

In [30]:
# Filtering Data
apps = apps[
    (apps["Rating"] >= 4.0) &
    (apps["Size"] >= 10) &
    (apps["Last Updated"].dt.month == 1)
    ]

In [31]:
# TOP 10
top_categories = (
    apps.groupby("Category")["Installs"].sum().nlargest(10).index
)
                                                           

In [32]:
chart_apps = (
    apps[apps["Category"].isin(top_categories)].groupby("Category",as_index=False).agg(
        Average_Rating = ("Rating","mean"),
        Total_Reviews = ("Reviews","sum"),
        Total_Installs = ("Installs","sum")
    )
)

In [33]:
chart_apps["Category"] = pd.Categorical(chart_apps["Category"],categories = top_categories,ordered = True)
chart_apps = chart_apps.sort_values("Total_Installs",ascending = False)

In [34]:
chart_apps["Total Reviews (Millions)"] = chart_apps["Total_Reviews"]/1_000_000

In [35]:
# Time Restriction
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

In [36]:
if current_time >= datetime.strptime("15:00","%H:%M").time() and current_time <= datetime.strptime("17:00","%H:%M").time():
    
    #Grouped Bar Chart
    fig = make_subplots(specs = [[{"secondary_y": True}]])
    
    #Average Rating
    fig.add_trace(
        go.Bar(
        x = chart_apps["Category"],
        y = chart_apps["Average_Rating"],
        name = "Average Rating",
        marker_color = "steelblue",
        offsetgroup = 1
        ),
        secondary_y = False
    
    )

    # Total Reviews
    fig.add_trace(
        go.Bar(
        x = chart_apps["Category"],
        y = chart_apps["Total Reviews (Millions)"],
        name = "Total Reviews (Millions)",
        marker_color = "darkorange",
        offsetgroup = 2
        ),
        secondary_y = True
    )
    
    fig.update_layout(
        title = "Average Rating and Total Reviews for Top 10 App Categories by Installs",
        xaxis_title = "App Category",
        yaxis_title = "Average Rating",
        barmode = "group",
        bargap = 0.25,
        bargroupgap = 0.1,
        template = "plotly_white",
        width = 1200,
        height = 650
    )
    fig.update_xaxes(tickangle = -25)
    
    fig.update_yaxes(title_text="Average Rating",range = [4.0,5.0],secondary_y = False)
    
    fig.update_yaxes(title_text="Total Reviews (Millions)",tickformat = ".1f",secondary_y = True)

    
    fig.show()
        

else:
    print("Grouped Bar Chart is only available between 3 PM IST and 5 PM IST")

    
    

Grouped Bar Chart is only available between 3 PM IST and 5 PM IST


In [ ]:
#CONCLUSION: 
#A grouped bar chart was created to compare the average rating and total review count of the top 10 app categories based on installs. 
#The data was filtered to include only categories with an average rating of at least 4.0, app size of at least 10 MB, and apps last updated in January. 
#The dashboard is configured to display this visualization only between 3:00 PM IST and 5:00 PM IST,
#ensuring that it meets all the specified assignment requirements.